02_Text_RAG.ipynb

1. Import Libraries

2. Load documents.pkl

3. Chunking

4. Load BGE-M3 Embeddings

5. Create FAISS

6. Save FAISS

7. Load Retriever

8. Load Mistral

9. Prompt Template

10. RAG Function

11. Test Questions

In [43]:
%pip install --upgrade langchain-huggingface
import pickle
import torch

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Libraries loaded!")

Note: you may need to restart the kernel to use updated packages.
Libraries loaded!


In [44]:
import pickle

with open("documents.pkl", "rb") as f:
    documents = pickle.load(f)

print(len(documents))

6


In [45]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Original Documents:", len(documents))
print("Chunks:", len(chunks))

print("\nExample Chunk:")
print(chunks[0].page_content[:800])

print("\nMetadata:")
print(chunks[0].metadata)

Original Documents: 6
Chunks: 65

Example Chunk:
Landmark Name: Abu Simbel Temples
Category: Rock-Cut Ancient Temple Complex / UNESCO World Heritage Site

Location:
{'city': 'Abu Simbel', 'governorate': 'Aswan', 'country': 'Egypt', 'region': 'Southern Upper Egypt, western shore of Lake Nasser', 'coordinates': {'latitude': 22.3372, 'longitude': 31.6258}}

Metadata:
{'landmark_id': 'abu_simbel'}


In [46]:
import torch
import transformers
import sentence_transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

Torch: 2.5.1+cu121
Transformers: 4.52.4
Sentence Transformers: 3.4.1


In [47]:
import sys

print(sys.executable)

c:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe


In [48]:
import transformers
import sentence_transformers

print(transformers.__file__)
print(transformers.__version__)

print(sentence_transformers.__file__)
print(sentence_transformers.__version__)

c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\__init__.py
4.52.4
c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\__init__.py
3.4.1


In [49]:
%pip show transformers

Name: transformers
Version: 4.52.4
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: sentence-transformers
Note: you may need to restart the kernel to use updated packages.


In [50]:
import torch
print(torch.__version__)

2.5.1+cu121


In [51]:
import sys
print(sys.executable)

c:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe


In [52]:
import sys

!{sys.executable} -m pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [53]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.5.1+cu121
True


In [54]:
import transformers
import sentence_transformers
import torch

print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Torch:", torch.__version__)

Transformers: 4.52.4
Sentence Transformers: 3.4.1
Torch: 2.5.1+cu121


In [55]:
import sys
import transformers
import sentence_transformers

print(sys.executable)
print(transformers.__file__)
print(sentence_transformers.__file__)

c:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe
c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\__init__.py
c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\__init__.py


In [56]:
import sys

!{sys.executable} -m pip uninstall -y transformers sentence-transformers
!{sys.executable} -m pip install transformers==4.52.4 sentence-transformers==3.4.1

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
Found existing installation: sentence-transformers 3.4.1
Uninstalling sentence-transformers-3.4.1:
  Successfully uninstalled sentence-transformers-3.4.1
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
  Using cached sentence_transformers-3.4.1-py3-none-any.whl.metadata (10 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)
Using cached sentence_transformers-3.4.1-py3-none-any.whl (275 kB)

   ---------------------------------------- 0/2 [transformers]
   ---------------------------------------- 0/2 [transformers]
   ---------------------------------------- 0/2 [transformers]
   ---------------------------------------- 0/2 [transformers]
   ---------------------------------------- 0/2 [transformers]
   ---------------------------------------- 0/2 [transformers]
   ---------------------------------------- 0/2 [transformers]
 

In [57]:
import transformers
import sentence_transformers

print(transformers.__version__)
print(sentence_transformers.__version__)

4.52.4
3.4.1


In [58]:
import transformers
import sentence_transformers
import torch

print(transformers.__version__)
print(sentence_transformers.__version__)
print(torch.__version__)

4.52.4
3.4.1
2.5.1+cu121


In [59]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model loaded successfully!")

Embedding model loaded successfully!


In [60]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

In [61]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [62]:
FAISS_PATH = "egypt_faiss"

vectorstore.save_local(FAISS_PATH)

print("FAISS saved!")

FAISS saved!


In [63]:
query = "Who built Abu Simbel temples?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(results):
    print("\nRESULT", i+1)
    print(doc.page_content[:500])
    print(doc.metadata)


RESULT 1
Question: How were the temples relocated?
Answer: Engineers cut both temples into more than 1,000 large blocks weighing 20 to 30 tonnes each, then reassembled them approximately 65 meters higher and 200 meters back from their original site, building an artificial mountain to replicate the original setting.

Question: Who built the Abu Simbel temples?
Answer: They were built by Pharaoh Ramesses II during the 19th Dynasty, around 1264 to 1244 BCE, to project Egyptian power toward Nubia and to hono
{'landmark_id': 'abu_simbel'}

RESULT 2
Question: How do I get to Abu Simbel?
Answer: Most visitors fly from Aswan, a flight of about 45 minutes, or take a road trip of roughly 3.5 to 4 hours each way. Flights and road travel can occasionally be disrupted by desert sandstorms, so it's worth checking conditions before your trip.

Question: Why were the Abu Simbel temples moved?
Answer: In the 1960s, the temples were relocated to higher ground in a UNESCO-led international engineering c

In [64]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [65]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [66]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [67]:

arabic_prompt = """
أنت مرشد سياحي ذكي ومتخصص في المعالم السياحية في مصر.

التعليمات:
- أجب باللغة العربية فقط.
- لا تستخدم أي كلمة أو جملة باللغة الإنجليزية إلا إذا كانت اسمًا رسميًا لمعلم أو مكان.
- اعتمد فقط على المعلومات الموجودة في السياق.
- إذا لم تجد الإجابة في السياق، قل:
  "عذرًا، لا أملك معلومات كافية للإجابة على هذا السؤال."
- اجعل الإجابة واضحة ومختصرة.

السياق:
{context}

السؤال:
{question}

الإجابة:
"""

In [68]:
english_prompt = """
You are an intelligent AI Tourism Guide specialized in Egyptian landmarks.

Instructions:
- Answer only in English.
- Use only the information provided in the context.
- Do not make up information.
- If the answer is not available in the context, reply:
  "Sorry, I don't have enough information to answer this question."
- Keep your answer clear and concise.

Context:
{context}

Question:
{question}

Answer:
"""

In [69]:
import sys
print(sys.executable)

c:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe


In [70]:
import sys

!{sys.executable} -m pip install langdetect

In [71]:
import re

def detect_language(text):
    if re.search(r'[\u0600-\u06FF]', text):
        return "ar"
    return "en"


def ask_rag(question):

    # Retrieve relevant documents
    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # Detect question language
    lang = detect_language(question)

    # Choose prompt
    if lang == "ar":
        selected_prompt = arabic_prompt
    else:
        selected_prompt = english_prompt

    # Build final prompt
    final_prompt = selected_prompt.format(
        context=context,
        question=question
    )

    # Generate answer
    answer = llm.invoke(final_prompt)

    return {
        "question": question,
        "language": lang,
        "answer": answer.content,
        "sources": [
            doc.metadata
            for doc in docs
        ]
    }

In [72]:
result = ask_rag(
    "Who built Abu Simbel?"
)

print(result["answer"])

print()

for source in result["sources"]:
    print(source)

 Pharaoh Ramesses II during the 19th Dynasty, around 1264 to 1244 BCE.

{'landmark_id': 'abu_simbel'}
{'landmark_id': 'abu_simbel'}
{'landmark_id': 'abu_simbel'}


In [73]:
result = ask_rag(
    "Where is Karnak Temple?"
)

print(result["answer"])

 Karnak Temple is located in Luxor, Egypt, on the East Bank of the Nile River.


In [74]:
result = ask_rag(
    "When was the Grand Egyptian Museum opened?"
)

print(result["answer"])

 The Grand Egyptian Museum held its full public grand opening on November 4, 2025.


In [75]:
result = ask_rag("مين بنى معبد أبو سمبل؟")

print(result["answer"])

 ربما بنيته الفاروق الكبير (Pharaoh Ramses II) في الحياة الخامسة من عصره.


In [76]:
result = ask_rag("أين يقع معبد الكرنك؟")

print(result["answer"])

 يقع معبد الكرنك في مدينة لوكسور، محافظة لوكسور، مصر.


In [77]:
vectorstore.save_local("egypt_faiss")
print("FAISS index saved successfully!")

FAISS index saved successfully!


In [78]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [79]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

In [80]:
import pickle

with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Chunks saved successfully!")

Chunks saved successfully!


In [81]:
import json

config = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "vectorstore": "egypt_faiss",
    "llm": "mistral",
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=4)

In [82]:
question = "من بنى معابد أبو سمبل؟"

docs = retriever.invoke(question)

print("Number of retrieved documents:", len(docs))

for i, doc in enumerate(docs, 1):
    print("=" * 80)
    print(f"Document {i}")
    print("=" * 80)
    print(doc.page_content)

Number of retrieved documents: 3
Document 1
Landmark Name: Abu Simbel Temples
Category: Rock-Cut Ancient Temple Complex / UNESCO World Heritage Site

Location:
{'city': 'Abu Simbel', 'governorate': 'Aswan', 'country': 'Egypt', 'region': 'Southern Upper Egypt, western shore of Lake Nasser', 'coordinates': {'latitude': 22.3372, 'longitude': 31.6258}}
Document 2
Landmark Name: Grand Egyptian Museum (GEM)
Category: Museum

Location:
{'city': 'Giza', 'governorate': 'Giza', 'country': 'Egypt', 'region': 'Greater Cairo, adjacent to the Giza Plateau', 'coordinates': {'latitude': 29.9925, 'longitude': 31.1131}}
Document 3
Landmark Name: Luxor Temple
Category: Ancient Temple / UNESCO World Heritage Site

Location:
{'city': 'Luxor', 'governorate': 'Luxor', 'country': 'Egypt', 'region': 'Upper Egypt (ancient Thebes, East Bank of the Nile, central Luxor city)', 'coordinates': {'latitude': 25.6997, 'longitude': 32.6396}}
